In [1]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

# Model Architecture

For the convolutional part we use the weights of the ResNet18 model. These convolutions will feed into a hidden layer with the output being a single logit with the threshold at 50%:

* below 0.5 - bonafide
* above 0.5 - spoofed

In [2]:
base_model = resnet18(weights=ResNet18_Weights.DEFAULT)
print(base_model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/toastu/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [6]:
class SpoofDetector(nn.Module):
    def __init__(self):
        super().__init__()
        
        
        self.network = resnet18(weights=ResNet18_Weights.DEFAULT)

        for param in self.network.parameters():
            param.requires_grad = False
        
        
        num_features = self.network.fc.in_features #512
        
        
        self.network.fc = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Dropout(p=0.5),           
            nn.Linear(128, 1)            
        )

    def forward(self, x):
        
        return self.network(x)

In [4]:
model = SpoofDetector()
print(model)

SpoofDetector(
  (network): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-0

Creating a custom class that will wrap our dataset and apply the necesary transform operations on the mel-spectogram image.

In [5]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms

In [6]:
class SpoofImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        """
        dataframe: The pandas DataFrame containing metadata (file_path, label, etc.)
        transform: The torchvision transforms to apply to the images
        """
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        
        return len(self.df)

    def __getitem__(self, idx):
        
        row = self.df.iloc[idx]
        img_path = row['img_path']
        label = row['label']
        speaker_id = row['speaker_id']
        
        system_id = "bonafide" if row['system_id'] == '-' else row['system_id']
        
        
        image = Image.open(img_path).convert('RGB')
        
        
        if self.transform is not None:
            image = self.transform(image)
            
        
        return image, torch.tensor(label, dtype=torch.float32), speaker_id, system_id

In [7]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),         
    transforms.Normalize(          
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

In [8]:
import pandas as pd

In [9]:
df = pd.read_csv("data/metadata_train.csv")

In [10]:
df

,speaker_id,system_id,label,img_path
0,LA_0098,A05,1,data/dataset_img/train/fake/sample_0.png
1,LA_0079,-,0,data/dataset_img/train/real/sample_1.png
2,LA_0089,-,0,data/dataset_img/train/real/sample_2.png
3,LA_0080,A03,1,data/dataset_img/train/fake/sample_3.png
4,LA_0088,A05,1,data/dataset_img/train/fake/sample_4.png
...,...,...,...,...
10075,LA_0098,A05,1,data/dataset_img/train/fake/sample_10075.png
10076,LA_0083,A02,1,data/dataset_img/train/fake/sample_10076.png
10077,LA_0096,A01,1,data/dataset_img/train/fake/sample_10077.png
10078,LA_0085,-,0,data/dataset_img/train/real/sample_10078.png


In [11]:

train_dataset = SpoofImageDataset(dataframe=df, transform=train_transforms)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)


images, labels, speaker_ids, system_ids = next(iter(train_loader))

print("--- Data Pipeline Diagnostic Test ---")
print(f"Batch Image Shape:  {images.shape}")  
print(f"Batch Label Shape:  {labels.shape}")  
print(f"First 5 Speaker IDs: {speaker_ids[:5]}")
print(f"First 5 System IDs:  {system_ids[:5]}")
print(f"First 5 Labels:      {labels[:5].tolist()}")

--- Data Pipeline Diagnostic Test ---
Batch Image Shape:  torch.Size([32, 3, 224, 224])
Batch Label Shape:  torch.Size([32])
First 5 Speaker IDs: ('LA_0093', 'LA_0083', 'LA_0083', 'LA_0088', 'LA_0082')
First 5 System IDs:  ('A04', 'bonafide', 'bonafide', 'A01', 'A01')
First 5 Labels:      [1.0, 0.0, 0.0, 1.0, 1.0]


### Training loop

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import DataLoader

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [14]:
df_dev = pd.read_csv("data/metadata_dev.csv")
df_dev

,speaker_id,system_id,label,img_path
0,LA_0069,-,0,data/dataset_img/dev/real/sample_0.png
1,LA_0069,-,0,data/dataset_img/dev/real/sample_1.png
2,LA_0069,-,0,data/dataset_img/dev/real/sample_2.png
3,LA_0069,-,0,data/dataset_img/dev/real/sample_3.png
4,LA_0069,-,0,data/dataset_img/dev/real/sample_4.png
...,...,...,...,...
24839,LA_0078,A06,1,data/dataset_img/dev/fake/sample_24839.png
24840,LA_0078,A06,1,data/dataset_img/dev/fake/sample_24840.png
24841,LA_0078,A06,1,data/dataset_img/dev/fake/sample_24841.png
24842,LA_0078,A06,1,data/dataset_img/dev/fake/sample_24842.png


In [15]:
val_dataset = SpoofImageDataset(dataframe=df_dev, transform=train_transforms)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [16]:
model = model.to(device)

In [20]:

#only passing the parameters from model with requires_grad = True to adam
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
num_epochs = 20


num_real = 2580
num_fake = 7500
weight_ratio = num_real / num_fake  # This will be roughly 0.344


pos_weight_tensor = torch.tensor([weight_ratio]).to(device)



criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

In [18]:
import time

In [21]:
best_val_loss = float('inf')
patience = 5  
patience_counter = 0

print("Starting Training...")

for epoch in range(num_epochs):
    start_time = time.time()
    
    
    model.train()  
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for images, labels, _, _ in train_loader:
        
        images, labels = images.to(device), labels.to(device)
        
        
        optimizer.zero_grad()
        
        
        outputs = model(images)
        
        
        outputs = outputs.squeeze(1)
        
        
        loss = criterion(outputs, labels)
        
        
        loss.backward()
        
    
        optimizer.step()
        
        
        train_loss += loss.item() * images.size(0)
        
        
        probs = torch.sigmoid(outputs)
        predictions = (probs >= 0.5).float()
        
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)
        
    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total

    
    model.eval()  
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    
    with torch.no_grad():
        for images, labels, _, _ in val_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            
            probs = torch.sigmoid(outputs)
            predictions = (probs >= 0.5).float()
            
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)
            
    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total
    
    epoch_time = time.time() - start_time
    
    
    print(f"Epoch {epoch+1:02d}/{num_epochs} | Time: {epoch_time:.0f}s")
    print(f"  Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
    print(f"  Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.4f}")
    
    
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), 'best_audio_spoof_model_weighted.pth')
        print("  -> New best model saved")
        patience_counter = 0  
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n Early stopping triggered. Validation loss hasn't improved in {patience} epochs.")
            break
    print("-" * 50)

print("Training Complete!")

Starting Training...
Epoch 01/20 | Time: 104s
  Train Loss: 0.0868 | Train Acc: 0.9311
  Val Loss:   0.0670 | Val Acc:   0.9608
  -> New best model saved
--------------------------------------------------
Epoch 02/20 | Time: 101s
  Train Loss: 0.0891 | Train Acc: 0.9265
  Val Loss:   0.0556 | Val Acc:   0.9549
  -> New best model saved
--------------------------------------------------
Epoch 03/20 | Time: 99s
  Train Loss: 0.0846 | Train Acc: 0.9298
  Val Loss:   0.0560 | Val Acc:   0.9562
--------------------------------------------------
Epoch 04/20 | Time: 100s
  Train Loss: 0.0847 | Train Acc: 0.9285
  Val Loss:   0.0590 | Val Acc:   0.9504
--------------------------------------------------
Epoch 05/20 | Time: 99s
  Train Loss: 0.0892 | Train Acc: 0.9264
  Val Loss:   0.0544 | Val Acc:   0.9515
  -> New best model saved
--------------------------------------------------
Epoch 06/20 | Time: 98s
  Train Loss: 0.0789 | Train Acc: 0.9322
  Val Loss:   0.0579 | Val Acc:   0.9637
-------

### Inference test on the unweighted model

In [19]:

df_eval = pd.read_csv("data/metadata_eval.csv")
eval_dataset = SpoofImageDataset(dataframe=df_eval, transform=train_transforms)
eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)


model.load_state_dict(torch.load('best_audio_spoof_model_unweighted.pth', map_location=device))
model.eval() 


all_labels = []
all_preds = []
all_speakers = []
all_systems = []

print("Running inference on Evaluation Dataset...")


with torch.no_grad():
    for images, labels, speaker_ids, system_ids in eval_loader:
        images = images.to(device)
        
       
        outputs = model(images).squeeze(1)
        probs = torch.sigmoid(outputs)
        predictions = (probs >= 0.5).float()
        
        
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predictions.cpu().numpy())
        all_speakers.extend(speaker_ids)
        all_systems.extend(system_ids)


df_results = pd.DataFrame({
    'Actual': all_labels,
    'Predicted': all_preds,
    'Speaker_ID': all_speakers,
    'System_ID': all_systems
})


total_samples = len(df_results)
correct_preds = (df_results['Actual'] == df_results['Predicted']).sum()
eval_accuracy = correct_preds / total_samples

print(f"\nFinal Evaluation Accuracy: {eval_accuracy:.4f} ({correct_preds}/{total_samples})")


false_positives = df_results[(df_results['Actual'] == 0) & (df_results['Predicted'] == 1)]


false_negatives = df_results[(df_results['Actual'] == 1) & (df_results['Predicted'] == 0)]

print(f"\nTotal False Positives (Flagged real humans): {len(false_positives)}")
print(f"Total False Negatives (Missed AI attacks): {len(false_negatives)}")

print("\n--- Top 3 AI Systems that fooled the model (False Negatives) ---")
print(false_negatives['System_ID'].value_counts().head(3))

print("\n--- Top 3 Human Speakers the model misidentified (False Positives) ---")
print(false_positives['Speaker_ID'].value_counts().head(3))

Running inference on Evaluation Dataset...

Final Evaluation Accuracy: 0.9021 (64260/71237)

Total False Positives (Flagged real humans): 1219
Total False Negatives (Missed AI attacks): 5758

--- Top 3 AI Systems that fooled the model (False Negatives) ---
System_ID
A17    3125
A18    1028
A19    1004
Name: count, dtype: int64

--- Top 3 Human Speakers the model misidentified (False Positives) ---
Speaker_ID
LA_0012    68
LA_0043    51
LA_0014    48
Name: count, dtype: int64


### Inference test on the weighted model

In [22]:

df_eval = pd.read_csv("data/metadata_eval.csv")
eval_dataset = SpoofImageDataset(dataframe=df_eval, transform=train_transforms)
eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)


model.load_state_dict(torch.load('best_audio_spoof_model_weighted.pth', map_location=device))
model.eval()


all_labels = []
all_preds = []
all_speakers = []
all_systems = []

print("Running inference on Evaluation Dataset...")


with torch.no_grad():
    for images, labels, speaker_ids, system_ids in eval_loader:
        images = images.to(device)
        
        
        outputs = model(images).squeeze(1)
        probs = torch.sigmoid(outputs)
        predictions = (probs >= 0.5).float()
        
        
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predictions.cpu().numpy())
        all_speakers.extend(speaker_ids)
        all_systems.extend(system_ids)


df_results = pd.DataFrame({
    'Actual': all_labels,
    'Predicted': all_preds,
    'Speaker_ID': all_speakers,
    'System_ID': all_systems
})


total_samples = len(df_results)
correct_preds = (df_results['Actual'] == df_results['Predicted']).sum()
eval_accuracy = correct_preds / total_samples

print(f"\nFinal Evaluation Accuracy: {eval_accuracy:.4f} ({correct_preds}/{total_samples})")


false_positives = df_results[(df_results['Actual'] == 0) & (df_results['Predicted'] == 1)]


false_negatives = df_results[(df_results['Actual'] == 1) & (df_results['Predicted'] == 0)]

print(f"\nTotal False Positives (Flagged real humans): {len(false_positives)}")
print(f"Total False Negatives (Missed AI attacks): {len(false_negatives)}")

print("\n--- Top 3 AI Systems that fooled the model (False Negatives) ---")
print(false_negatives['System_ID'].value_counts().head(3))

print("\n--- Top 3 Human Speakers the model misidentified (False Positives) ---")
print(false_positives['Speaker_ID'].value_counts().head(3))

Running inference on Evaluation Dataset...

Final Evaluation Accuracy: 0.8805 (62725/71237)

Total False Positives (Flagged real humans): 693
Total False Negatives (Missed AI attacks): 7819

--- Top 3 AI Systems that fooled the model (False Negatives) ---
System_ID
A17    3690
A18    1762
A19    1514
Name: count, dtype: int64

--- Top 3 Human Speakers the model misidentified (False Positives) ---
Speaker_ID
LA_0038    36
LA_0012    36
LA_0041    27
Name: count, dtype: int64


### Results

As it looks, the unweighted model performs better than the weighted model if we just look at the raw data, however the weighted model, while it performed poorly, it almost halved the number of false positives. This creates two models that specialize in different thigns:

The unweighted model could be used in something like alert systems where a missed attack is much more damaging than a false positive

The weighted model could be used in something like an authentication app, where flagging a real user as an ai will lock them out of the account, in the real world, this might mean the users will hate the app or we could even face legal issues.

### Dataset problem

Anyhow, neither of the two models have great looking numbers, their performance is in fact quite terrible compared to real world standards. But that s ok, they are just a proof of concept. In fact i bet the models will do even worse if they are to be implemented in a modern app.

The reason for this is the dataset: ASVSpoof is a very old dataset (2019), back then even humans could easily tell apart a human voice from a robot voice. Nowadays with tools like ElevenLabs that are so good and cutting edge even humans have a hard time detecting  the difference, it will be impossible for this model to actually be reliable.

I simply didnt have access to a more recent dataset and didnt have the time and resources to make my own hence why i deliberatly possitioned this as a proof of concept and a learning experience rather than a production ready system.

### Possible solutions

For this sort of problem, academic datasets like ASVSpoof are definetly not realiable. There needs to be a dataset made from scratch, perhaps from real world vishing attacks or synthetically made by recording real people and clonign their voices with ElevenLabs or other tools.

Also we would probably need to look even deeper into the mel-spectograms for artifacts since the current model possibly just looks for the speach pattern or inconsisten frequencies.


In [7]:
import os
import pandas as pd

base_path = "/home/toastu/Projects/Bit/data/archive/LA/LA/"
dev_protocol_path = os.path.join(base_path, "ASVspoof2019_LA_cm_protocols", "ASVspoof2019.LA.cm.dev.trl.txt")
dev_audio_folder = os.path.join(base_path, "ASVspoof2019_LA_dev", "flac")

# collumns based on the README of the original dataset
columns = ["speaker_id", "audio_file_name", "ignored", "system_id", "label"]

df_dev = pd.read_csv(dev_protocol_path, sep=" ", header=None, names=columns)

df_dev = df_dev.drop(columns=["ignored"])


df_dev['file_path'] = df_dev['audio_file_name'].apply(lambda x: os.path.join(dev_audio_folder, x + ".flac"))

df_dev.sample(10)

,speaker_id,audio_file_name,system_id,label,file_path
21527,LA_0069,LA_D_9775299,A06,spoof,/home/toastu/Projects/Bit/data/archive/LA/LA/A...
15942,LA_0075,LA_D_2653693,A04,spoof,/home/toastu/Projects/Bit/data/archive/LA/LA/A...
19730,LA_0075,LA_D_4054212,A05,spoof,/home/toastu/Projects/Bit/data/archive/LA/LA/A...
13880,LA_0069,LA_D_4954418,A04,spoof,/home/toastu/Projects/Bit/data/archive/LA/LA/A...
17104,LA_0078,LA_D_3484420,A04,spoof,/home/toastu/Projects/Bit/data/archive/LA/LA/A...
16391,LA_0076,LA_D_5234827,A04,spoof,/home/toastu/Projects/Bit/data/archive/LA/LA/A...
4643,LA_0074,LA_D_8348784,A01,spoof,/home/toastu/Projects/Bit/data/archive/LA/LA/A...
5708,LA_0077,LA_D_7038850,A01,spoof,/home/toastu/Projects/Bit/data/archive/LA/LA/A...
9661,LA_0078,LA_D_3318751,A02,spoof,/home/toastu/Projects/Bit/data/archive/LA/LA/A...
17499,LA_0069,LA_D_2946880,A05,spoof,/home/toastu/Projects/Bit/data/archive/LA/LA/A...


In [9]:
import torch
import io
import librosa
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torchvision.transforms as transforms



print("--- Running Truth Serum Diagnostic ---")

device = 'cuda'
test_model = SpoofDetector().to(device)

try:
    
    test_model.load_state_dict(torch.load('best_audio_spoof_model_unweighted.pth', map_location=device))
    test_model.eval()
    print("Weights loaded successfully!")
except Exception as e:
    print(f"FAILED TO LOAD WEIGHTS: {e}")


def web_app_pipeline(file_path):
    y, sr = librosa.load(file_path, sr=16000)
    y_fixed = librosa.util.fix_length(data=y, size=4*16000)
    mel_spec = librosa.feature.melspectrogram(y=y_fixed, sr=sr, n_mels=128)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    buf = io.BytesIO()
    plt.imsave(buf, mel_spec_db, cmap='magma', origin='lower', format='png')
    buf.seek(0)
    img = Image.open(buf).convert('RGB')
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return transform(img).unsqueeze(0).to(device)


sample_fake_path = df_dev[df_dev['label'] == "spoof"].iloc[0]['file_path']
tensor = web_app_pipeline(sample_fake_path)

with torch.no_grad():
    output = test_model(tensor).squeeze(1)
    prob = torch.sigmoid(output).item()
    print(f"\nTarget: SPOOF (Expected Raw Probability > 0.8000)")
    print(f"Actual Raw Probability: {prob:.4f}")

--- Running Truth Serum Diagnostic ---
Weights loaded successfully!

Target: SPOOF (Expected Raw Probability > 0.8000)
Actual Raw Probability: 0.9982
